### breast_cancer.py

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

data = load_breast_cancer()
X = data.data
y = data.target
labels = [0, 1]
target_names = list(data.target_names)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== Task (a): Confusion Matrix (threshold=0.5) ===")
cm = confusion_matrix(y_test, y_pred, labels=labels)
print("Rows = actual, columns = predicted, labels = [malignant, benign]")
print(cm)
print("\nInterpretation:")
print(f"Malignant correctly predicted: {cm[0, 0]}")
print(f"Malignant predicted as benign: {cm[0, 1]} (critical miss)")
print(f"Benign predicted as malignant: {cm[1, 0]}")
print(f"Benign correctly predicted: {cm[1, 1]}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

print("\n=== Task (b): Malignant Threshold = 0.3 ===")
malignant_class = 0
benign_class = 1
classes = model.named_steps["logisticregression"].classes_
malignant_proba_index = np.where(classes == malignant_class)[0][0]
malignant_proba = model.predict_proba(X_test)[:, malignant_proba_index]
y_pred_03 = np.where(malignant_proba >= 0.3, malignant_class, benign_class)
cm_03 = confusion_matrix(y_test, y_pred_03, labels=labels)
print("Rows = actual, columns = predicted, labels = [malignant, benign]")
print(cm_03)
print("\nClassification Report (malignant threshold=0.3):")
print(classification_report(y_test, y_pred_03, target_names=target_names))

recall_05_mal = cm[0, 0] / cm[0].sum()
recall_03_mal = cm_03[0, 0] / cm_03[0].sum()
recall_05_ben = cm[1, 1] / cm[1].sum()
recall_03_ben = cm_03[1, 1] / cm_03[1].sum()
print("\nRecall comparison:")
print(f"  Malignant (class 0): {recall_05_mal:.4f} -> {recall_03_mal:.4f} (change: {recall_03_mal - recall_05_mal:+.4f})")
print(f"  Benign (class 1):    {recall_05_ben:.4f} -> {recall_03_ben:.4f} (change: {recall_03_ben - recall_05_ben:+.4f})")
print("-> Lowering the malignant threshold flags more cases as malignant, which prioritizes catching critical cases.")

print("\n=== Task (c): max_iter=10 (very low) ===")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    low_iter_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=10, random_state=RANDOM_STATE)
    )
    low_iter_model.fit(X_train, y_train)

convergence_warnings = [
    warning for warning in caught
    if issubclass(warning.category, ConvergenceWarning)
]
if convergence_warnings:
    print(f"Warning: {convergence_warnings[0].message}")
    print("\nMeaning: The optimizer did not converge in 10 iterations.")
    print("-> Model coefficients may not be optimal. Increase max_iter for more reliable training.")
else:
    print("No convergence warning for this scaled split, but 10 iterations is still an unsafe training budget.")

print("\n=== Task (d): ROC Curve ===")
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_estimator(
    model,
    X_test,
    y_test,
    pos_label=malignant_class,
    name="Malignant",
    ax=ax
)
ax.plot([0, 1], [0, 1], "k--", label="Random Classifier")
ax.set_title("ROC Curve - Breast Cancer Classification")
ax.legend()
plt.tight_layout()
roc_path = OUTPUT_DIR / "roc_curve_breast_cancer.png"
plt.savefig(roc_path, dpi=150)
plt.show()
print(f"ROC curve saved as '{roc_path}'")
